In [143]:
from scipy.sparse import kron, eye
from scipy.sparse import diags
import numpy as np

In [144]:
dim = 5
N = 8

# A

In [145]:
# create stiffness matrix for 1d: L
# embed it into higher dim with kron(I, L)
# in each step: add kron(L_1d, I), where L_1d in of size (n,n)
main_diag = 2 * np.ones(N)
off_diag = -1 * np.ones(N - 1)
L = diags([off_diag, main_diag, off_diag], offsets=[-1, 0, 1], shape=(N, N), dtype=int)
I = eye(N, dtype=int)
A = L
for d in range(1,dim):
    A = kron(I, A) + kron(L, eye(N**d, dtype=int))
A = A.toarray()
print(A)  # This is the 2D Poisson stiffness matrix

[[10 -1  0 ...  0  0  0]
 [-1 10 -1 ...  0  0  0]
 [ 0 -1 10 ...  0  0  0]
 ...
 [ 0  0  0 ... 10 -1  0]
 [ 0  0  0 ... -1 10 -1]
 [ 0  0  0 ...  0 -1 10]]


In [146]:
A.shape

(32768, 32768)

# B

In [147]:
def to_nd_from_1d(i):
    k = np.zeros([dim], dtype=int)
    i_rem = i
    for j in range(dim):
        kj = i_rem // N**(dim-(j+1))
        i_rem -= kj * N**(dim-(j+1))
        k[j] = kj
    return k

In [148]:
#to_nd_from_1d(9)

In [149]:
def to_1d_from_nd(k):
    return np.dot(k, N**np.arange(dim-1,-1,-1))

In [150]:
#to_1d_from_nd(np.array([1,1]))

In [151]:
#dim = 2
#N = 4
#
n = N**dim
indx_min = 0
indx_max = N-1
B = np.zeros((n,n), dtype=int)
for i in range(n): # go ever each row of the stiffness matrix
    k = to_nd_from_1d(i)
    B[i,i] = 2*dim
    for d in range(dim):
        if k[d]+1 <= indx_max:
            B[i,i+N**(dim-(d+1))] = -1
        if k[d]-1 >= indx_min:
            B[i,i-N**(dim-(d+1))] = -1
B

array([[10, -1,  0, ...,  0,  0,  0],
       [-1, 10, -1, ...,  0,  0,  0],
       [ 0, -1, 10, ...,  0,  0,  0],
       ...,
       [ 0,  0,  0, ..., 10, -1,  0],
       [ 0,  0,  0, ..., -1, 10, -1],
       [ 0,  0,  0, ...,  0, -1, 10]], shape=(32768, 32768))

In [152]:
A.shape

(32768, 32768)

# compare A & B

In [ ]:
np.nonzero(A - B)

(array([], dtype=int64), array([], dtype=int64))

: 